# Feature Engineering
**CareGuard — Hospital Readmission Risk Prediction**

This notebook prepares the raw diabetes dataset for model training by cleaning, encoding, and engineering features.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Step 1 — Load Data
Read the raw CSV, treating `?` as missing values. Create a binary target variable: `1` if the patient was readmitted within 30 days, `0` otherwise.

In [2]:
df = pd.read_csv('../data/raw/dataset_diabetes/diabetic_data.csv', na_values='?')

df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

print(f'Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Target distribution:\n{df["readmitted_30"].value_counts()}')

Loaded dataset: 101,766 rows, 51 columns
Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64


C:\Users\aisid\AppData\Local\Temp\ipykernel_6868\2119892560.py:1: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/dataset_diabetes/diabetic_data.csv', na_values='?')


## Step 2 — Drop High-Missing / Non-Informative Columns
Remove columns with excessive missing values or identifiers that carry no predictive signal.

In [3]:
cols_to_drop = ['weight', 'payer_code', 'encounter_id', 'patient_nbr', 'medical_specialty']
df.drop(columns=cols_to_drop, inplace=True)

print(f'Shape after dropping columns: {df.shape}')

Shape after dropping columns: (101766, 46)


## Step 3 — Handle Missing Values
Drop rows where `race` or `gender` is missing, as these are small proportions and imputation would be misleading.

In [4]:
before = len(df)
df.dropna(subset=['race', 'gender'], inplace=True)
after = len(df)

print(f'Dropped {before - after:,} rows with missing race/gender')
print(f'Shape after dropping: {df.shape}')

Dropped 2,273 rows with missing race/gender
Shape after dropping: (99493, 46)


## Step 4 — Encode Age
Map age bracket strings to ordinal integers representing decade bands (0–9).

In [5]:
age_map = {
    '[0-10)':  0,
    '[10-20)': 1,
    '[20-30)': 2,
    '[30-40)': 3,
    '[40-50)': 4,
    '[50-60)': 5,
    '[60-70)': 6,
    '[70-80)': 7,
    '[80-90)': 8,
    '[90-100)': 9
}

df['age'] = df['age'].map(age_map)

print('Age encoding complete')
print(df['age'].value_counts().sort_index())

Age encoding complete
age
0      160
1      682
2     1611
3     3699
4     9465
5    16895
6    21988
7    25469
8    16800
9     2724
Name: count, dtype: int64


## Step 5 — Encode Gender
Binary encode gender: `Male=1`, `Female=0`. Rows with unexpected values are dropped.

In [6]:
df = df[df['gender'].isin(['Male', 'Female'])]
df['gender'] = (df['gender'] == 'Male').astype(int)

print('Gender encoding complete')
print(df['gender'].value_counts())

Gender encoding complete
gender
0    53575
1    45917
Name: count, dtype: int64


## Step 6 — One-Hot Encode Race
Expand the `race` column into binary indicator columns, dropping the first to avoid multicollinearity.

In [7]:
race_dummies = pd.get_dummies(df['race'], prefix='race', drop_first=True)
df = pd.concat([df.drop(columns='race'), race_dummies], axis=1)

print('Race one-hot encoding complete')
print('New race columns:', list(race_dummies.columns))

Race one-hot encoding complete
New race columns: ['race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other']


## Step 7 — Engineer `prior_visits` Feature
Aggregate prior healthcare utilisation into a single feature: sum of inpatient, outpatient, and emergency visits.

In [8]:
df['prior_visits'] = (
    df['number_inpatient'] +
    df['number_outpatient'] +
    df['number_emergency']
)

print('prior_visits feature created')
print(df['prior_visits'].describe())

prior_visits feature created
count    99492.000000
mean         1.217806
std          2.308690
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max         80.000000
Name: prior_visits, dtype: float64


## Step 8 — Encode Medication Change Columns
For each medication column, encode `'Ch'` (dosage changed) as `1` and all other values as `0`.

In [9]:
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'insulin', 'tolazamide'
]

for col in med_cols:
    if col in df.columns:
        df[col] = (df[col] == 'Ch').astype(int)

print('Medication change columns encoded')
print(df[med_cols].sum().sort_values(ascending=False))

Medication change columns encoded
metformin         0
repaglinide       0
nateglinide       0
chlorpropamide    0
glimepiride       0
glipizide         0
glyburide         0
pioglitazone      0
rosiglitazone     0
acarbose          0
insulin           0
tolazamide        0
dtype: int64


## Step 9 — Encode `change` and `diabetesMed`
- `change`: `'Ch'` → `1`, else `0`
- `diabetesMed`: `'Yes'` → `1`, else `0`

In [10]:
df['change'] = (df['change'] == 'Ch').astype(int)
df['diabetesMed'] = (df['diabetesMed'] == 'Yes').astype(int)

print('change and diabetesMed encoded')
print(df[['change', 'diabetesMed']].value_counts())

change and diabetesMed encoded
change  diabetesMed
1       1              45910
0       1              30581
        0              23001
Name: count, dtype: int64


## Step 10 — Drop Original `readmitted` and Remaining String Columns
Remove the original target column and any remaining object-type columns that were not encoded.

In [11]:
df.drop(columns=['readmitted'], inplace=True)

remaining_str_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Remaining string columns to drop: {remaining_str_cols}')
df.drop(columns=remaining_str_cols, inplace=True)

print(f'Shape after dropping string columns: {df.shape}')

Remaining string columns to drop: ['diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'acetohexamide', 'tolbutamide', 'miglitol', 'troglitazone', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Shape after dropping string columns: (99492, 33)


C:\Users\aisid\AppData\Local\Temp\ipykernel_6868\2531901177.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  remaining_str_cols = df.select_dtypes(include='object').columns.tolist()


## Step 11 — Save Processed Data
Write the engineered feature set to `data/processed/features.csv`.

In [12]:
output_path = '../data/processed/features.csv'
df.to_csv(output_path, index=False)

print(f'Saved processed data to {output_path}')

Saved processed data to ../data/processed/features.csv


## Step 12 — Final Summary

In [13]:
print(f'Final dataset shape: {df.shape}')
print(f'\nTarget balance:')
print(df['readmitted_30'].value_counts(normalize=True).rename({0: 'Not readmitted <30d', 1: 'Readmitted <30d'}))
print(f'\nFirst few rows:')
df.head()

Final dataset shape: (99492, 33)

Target balance:
readmitted_30
Not readmitted <30d    0.88774
Readmitted <30d        0.11226
Name: proportion, dtype: float64

First few rows:


,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,tolazamide,insulin,change,diabetesMed,readmitted_30,race_Asian,race_Caucasian,race_Hispanic,race_Other,prior_visits
0,0,0,6,25,1,1,41,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,False,True,False,False,0
1,0,1,1,1,7,3,59,0,18,0,0,0,9,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,False,True,False,False,0
2,0,2,1,1,7,2,11,5,13,2,0,1,6,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,False,False,False,False,3
3,1,3,1,1,7,2,44,1,16,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,False,True,False,False,0
4,1,4,1,1,7,1,51,0,8,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,False,True,False,False,0
